# 06y promo split 260520

Split the PUBLIC 06x expanded dataset into `is_promotion = 0` and `is_promotion = 1` datasets, keeping row lineage and validation outputs under PUBLIC.

## 0. Setup and PUBLIC paths


In [1]:
from pathlib import Path
import hashlib
import json
import subprocess
import zipfile

import pandas as pd

STEP = '_06y_promo_split_260520'
TODAY = '2026-05-20'
ROOT = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip()).resolve()
PUBLIC = (ROOT / 'PUBLIC').resolve()
SOURCE_DATASET = PUBLIC / 'results' / '_06x_dataset_generation_260515' / '06x_expanded_dataset.csv'
OUT_DIR = PUBLIC / 'results' / STEP
NOTEBOOK_PATH = PUBLIC / 'notebooks' / '06y_promo_split_260520.ipynb'
NOTE_PATH = PUBLIC / 'note.md'
ZIP_PATH = PUBLIC / 'zip' / f'{STEP}_review_package.zip'

PROMO0_PATH = OUT_DIR / '06y_expanded_dataset_promo_0.csv'
PROMO1_PATH = OUT_DIR / '06y_expanded_dataset_promo_1.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)


## 1. Helper functions and input loading


In [2]:
def inside_public(path):
    resolved = Path(path).resolve()
    return resolved == PUBLIC or PUBLIC in resolved.parents

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def write_csv(df, name):
    path = OUT_DIR / name
    df.to_csv(path, index=False, encoding='utf-8-sig')
    return path

def passfail(ok):
    return 'PASS' if bool(ok) else 'FAIL'

if not SOURCE_DATASET.exists():
    raise SystemExit(f'missing source dataset: {SOURCE_DATASET}')

source_hash_before = sha256_file(SOURCE_DATASET)
source_mtime_before = SOURCE_DATASET.stat().st_mtime
data = pd.read_csv(SOURCE_DATASET)

if 'is_promotion' not in data.columns:
    raise SystemExit('missing required column: is_promotion')


## 2. Source profile


In [3]:
promo_counts = data['is_promotion'].value_counts(dropna=False).sort_index()
source_profile = pd.DataFrame([
    {'metric': 'source_file', 'value': str(SOURCE_DATASET.relative_to(PUBLIC))},
    {'metric': 'source_row_count', 'value': len(data)},
    {'metric': 'source_column_count', 'value': len(data.columns)},
    {'metric': 'is_promotion_distribution', 'value': json.dumps(promo_counts.to_dict(), ensure_ascii=False)},
    {'metric': 'target_distribution', 'value': json.dumps(data['is_repurchase'].value_counts(dropna=False).to_dict(), ensure_ascii=False) if 'is_repurchase' in data.columns else 'missing'},
])
write_csv(source_profile, '06y_source_profile.csv')


WindowsPath('C:/Code/ott-churn-prediction/PUBLIC/results/_06y_promo_split_260520/06y_source_profile.csv')

## 3. Split by is_promotion


In [4]:
promo_numeric = pd.to_numeric(data['is_promotion'], errors='coerce')
promo0 = data.loc[promo_numeric.eq(0)].copy()
promo1 = data.loc[promo_numeric.eq(1)].copy()
unexpected = data.loc[~promo_numeric.isin([0, 1])].copy()

promo0.to_csv(PROMO0_PATH, index=False, encoding='utf-8-sig')
promo1.to_csv(PROMO1_PATH, index=False, encoding='utf-8-sig')

split_summary = pd.DataFrame([
    {'split_name': 'source', 'file_name': SOURCE_DATASET.name, 'row_count': len(data), 'column_count': len(data.columns), 'is_promotion_unique_values': json.dumps(sorted(data['is_promotion'].dropna().astype(str).unique().tolist()), ensure_ascii=False)},
    {'split_name': 'promo_0', 'file_name': PROMO0_PATH.name, 'row_count': len(promo0), 'column_count': len(promo0.columns), 'is_promotion_unique_values': json.dumps(sorted(promo0['is_promotion'].dropna().astype(str).unique().tolist()), ensure_ascii=False)},
    {'split_name': 'promo_1', 'file_name': PROMO1_PATH.name, 'row_count': len(promo1), 'column_count': len(promo1.columns), 'is_promotion_unique_values': json.dumps(sorted(promo1['is_promotion'].dropna().astype(str).unique().tolist()), ensure_ascii=False)},
    {'split_name': 'unexpected', 'file_name': '', 'row_count': len(unexpected), 'column_count': len(unexpected.columns), 'is_promotion_unique_values': json.dumps(sorted(unexpected['is_promotion'].dropna().astype(str).unique().tolist()), ensure_ascii=False)},
])
write_csv(split_summary, '06y_promo_split_summary.csv')


WindowsPath('C:/Code/ott-churn-prediction/PUBLIC/results/_06y_promo_split_260520/06y_promo_split_summary.csv')

## 4. Schema and target distribution checks


In [5]:
schema_rows = []
for col in data.columns:
    schema_rows.append({
        'column_name': col,
        'source_dtype': str(data[col].dtype),
        'promo0_dtype': str(promo0[col].dtype),
        'promo1_dtype': str(promo1[col].dtype),
        'source_missing_count': int(data[col].isna().sum()),
        'promo0_missing_count': int(promo0[col].isna().sum()),
        'promo1_missing_count': int(promo1[col].isna().sum()),
    })
write_csv(pd.DataFrame(schema_rows), '06y_schema_check.csv')

target_rows = []
if 'is_repurchase' in data.columns:
    for name, df in [('source', data), ('promo_0', promo0), ('promo_1', promo1)]:
        counts = df['is_repurchase'].value_counts(dropna=False).to_dict()
        target_rows.append({'split_name': name, 'row_count': len(df), 'target_distribution': json.dumps(counts, ensure_ascii=False)})
write_csv(pd.DataFrame(target_rows), '06y_target_distribution_by_split.csv')


WindowsPath('C:/Code/ott-churn-prediction/PUBLIC/results/_06y_promo_split_260520/06y_target_distribution_by_split.csv')

## 5. README and note update


In [6]:
readme = f'''# 06y_promo_split_260520

## Purpose
Split the PUBLIC 06x expanded dataset into two files using `is_promotion`.

## Source
- `{SOURCE_DATASET.relative_to(PUBLIC)}`

## Outputs
- `{PROMO0_PATH.relative_to(PUBLIC)}`: rows where `is_promotion == 0`
- `{PROMO1_PATH.relative_to(PUBLIC)}`: rows where `is_promotion == 1`

## Validation
- Source rows: {len(data)}
- promo_0 rows: {len(promo0)}
- promo_1 rows: {len(promo1)}
- unexpected `is_promotion` rows: {len(unexpected)}
- Column order is preserved from the source dataset.
- Source dataset hash and mtime are checked before and after writing outputs.
'''
(OUT_DIR / 'README.md').write_text(readme, encoding='utf-8')

note_block = f'''

## {TODAY} 06y_promo_split_260520
- PUBLIC 06x expanded dataset을 `is_promotion` 기준으로 분할함.
- source rows: {len(data)}.
- promo_0 rows: {len(promo0)}.
- promo_1 rows: {len(promo1)}.
- unexpected is_promotion rows: {len(unexpected)}.
- outputs: PUBLIC/results/_06y_promo_split_260520.
'''
note_text = NOTE_PATH.read_text(encoding='utf-8') if NOTE_PATH.exists() else ''
if '## 2026-05-20 06y_promo_split_260520' not in note_text:
    with NOTE_PATH.open('a', encoding='utf-8') as f:
        f.write(note_block)
note_text = NOTE_PATH.read_text(encoding='utf-8')
(OUT_DIR / 'note_tail_copy.md').write_text('\n'.join(note_text.splitlines()[-120:]) + '\n', encoding='utf-8')


3729

## 6. Final checks and print summary


In [7]:
source_hash_after = sha256_file(SOURCE_DATASET)
source_mtime_after = SOURCE_DATASET.stat().st_mtime

checks = []
def add_check(check, ok, detail=''):
    checks.append({'check': check, 'status': passfail(ok), 'detail': detail})

add_check('paths_resolve_inside_PUBLIC', all(inside_public(p) for p in [SOURCE_DATASET, OUT_DIR, NOTEBOOK_PATH, NOTE_PATH, ZIP_PATH]), '')
add_check('source_dataset_not_modified', source_hash_before == source_hash_after and source_mtime_before == source_mtime_after, 'SHA256 and mtime unchanged')
add_check('notebook_exists', NOTEBOOK_PATH.exists(), str(NOTEBOOK_PATH))
add_check('source_has_is_promotion', 'is_promotion' in data.columns, '')
add_check('source_has_is_repurchase', 'is_repurchase' in data.columns, '')
add_check('unexpected_is_promotion_rows_zero', len(unexpected) == 0, str(len(unexpected)))
add_check('split_rows_sum_to_source', len(promo0) + len(promo1) == len(data), f'{len(promo0)} + {len(promo1)} = {len(data)}')
add_check('promo0_only_has_zero', set(pd.to_numeric(promo0['is_promotion'], errors='coerce').dropna().unique().tolist()) == {0}, str(sorted(promo0['is_promotion'].dropna().astype(str).unique().tolist())))
add_check('promo1_only_has_one', set(pd.to_numeric(promo1['is_promotion'], errors='coerce').dropna().unique().tolist()) == {1}, str(sorted(promo1['is_promotion'].dropna().astype(str).unique().tolist())))
add_check('promo0_columns_match_source', promo0.columns.tolist() == data.columns.tolist(), '')
add_check('promo1_columns_match_source', promo1.columns.tolist() == data.columns.tolist(), '')
add_check('promo0_file_created', PROMO0_PATH.exists(), str(PROMO0_PATH))
add_check('promo1_file_created', PROMO1_PATH.exists(), str(PROMO1_PATH))
add_check('README_created', (OUT_DIR / 'README.md').exists(), '')
add_check('note_md_updated', '06y_promo_split_260520' in NOTE_PATH.read_text(encoding='utf-8'), '')

fail_count = sum(1 for row in checks if row['status'] == 'FAIL')
checks.append({'check': 'critical_fail_count_zero', 'status': passfail(fail_count == 0), 'detail': str(fail_count)})
write_csv(pd.DataFrame(checks), '06y_final_checks.csv')

final_status = 'PASS' if fail_count == 0 else 'FAIL'
summary_lines = [
    'PRINT_VALIDATION_SUMMARY_START',
    'step=06y_promo_split_260520',
    f'source_rows={len(data)}',
    f'promo_0_rows={len(promo0)}',
    f'promo_1_rows={len(promo1)}',
    f'unexpected_is_promotion_rows={len(unexpected)}',
    f'source_columns={len(data.columns)}',
    f'final_status={final_status}',
    'PRINT_VALIDATION_SUMMARY_END',
]
summary_text = '\n'.join(summary_lines) + '\n'
(OUT_DIR / '06y_promo_split_print_summary.txt').write_text(summary_text, encoding='utf-8')
print(summary_text, end='')


PRINT_VALIDATION_SUMMARY_START
step=06y_promo_split_260520
source_rows=23097
promo_0_rows=11193
promo_1_rows=11904
unexpected_is_promotion_rows=0
source_columns=82
final_status=PASS
PRINT_VALIDATION_SUMMARY_END


## 7. PUBLIC review zip package


In [8]:
def create_zip():
    if ZIP_PATH.exists():
        ZIP_PATH.unlink()
    with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        z.write(NOTEBOOK_PATH, arcname=NOTEBOOK_PATH.relative_to(PUBLIC).as_posix())
        for p in sorted(OUT_DIR.rglob('*')):
            if p.is_file():
                z.write(p, arcname=p.relative_to(PUBLIC).as_posix())

create_zip()
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    inventory = pd.DataFrame([{'zip_member': info.filename, 'file_size': info.file_size, 'compress_size': info.compress_size} for info in z.infolist()])
write_csv(inventory, '06y_review_zip_inventory.csv')
create_zip()
